### Generating text with a pretrained llm

In [1]:
from pathlib import Path
import sys

ROOT_DIR = Path.cwd().parent  # Get parent of current directory
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import torch
from safetensors.torch import load_file

from base_model.qwen import (
    QWEN_CONFIG_06_B,
    Qwen3Model,
    Qwen3Tokenizer,
    generate_text,
    load_hf_weights_into_qwen,
)


#def main(prompt):
model_dir = Path.cwd() / "qwen"
tokenizer = Qwen3Tokenizer(model_dir / "tokenizer.json")
weights = load_file(model_dir / "model.safetensors")

model = Qwen3Model(QWEN_CONFIG_06_B)
load_hf_weights_into_qwen(
    model,
    param_config={
        "n_layers": QWEN_CONFIG_06_B["n_layers"],
        "hidden_dim": QWEN_CONFIG_06_B["hidden_dim"],
    },
    params=weights,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.to(torch.bfloat16)



Qwen3Model(
  (tok_emb): Embedding(151936, 1024)
  (trf_blocks): ModuleList(
    (0-27): 28 x TransformerBlock(
      (att): GroupQueryAttention(
        (w_query): Linear(in_features=1024, out_features=2048, bias=False)
        (w_keys): Linear(in_features=1024, out_features=1024, bias=False)
        (w_values): Linear(in_features=1024, out_features=1024, bias=False)
        (proj_out): Linear(in_features=2048, out_features=1024, bias=False)
        (q_norm): RMSNorm()
        (k_norm): RMSNorm()
      )
      (ff): FeedForward(
        (fc1): Linear(in_features=1024, out_features=3072, bias=False)
        (fc2): Linear(in_features=1024, out_features=3072, bias=False)
        (fc3): Linear(in_features=3072, out_features=1024, bias=False)
      )
      (rms_norm1): RMSNorm()
      (rms_norm2): RMSNorm()
    )
  )
  (final_norm): RMSNorm()
  (out_head): Linear(in_features=1024, out_features=151936, bias=False)
)

In [2]:
"""response = generate_text(
    model,
    tokenizer,
    prompt="What is the capital of france",
    max_new_tokens=128,
    temperature=0,
    chat=True,
    enable_thinking=False,
)
print(response)"""

'response = generate_text(\n    model,\n    tokenizer,\n    prompt="What is the capital of france",\n    max_new_tokens=128,\n    temperature=0,\n    chat=True,\n    enable_thinking=False,\n)\nprint(response)'

In [3]:
import torch
@torch.inference_mode()
def generate_text_basic_stream(model, token_ids, max_new_tokens, eos_token_id=None):
    model.eval()

    for _ in range(max_new_tokens):
        out = model(token_ids)[:, -1]
        next_token = torch.argmax(out, dim=-1, keepdim=True)

        if (eos_token_id is not None 
            and torch.all(next_token == eos_token_id)):
            break

        yield next_token

        token_ids = torch.cat([token_ids, next_token], dim=1)


In [4]:
prompt = "explain machine learning technology"

input_tokens = torch.tensor(tokenizer.encode(prompt), device=device).unsqueeze(0)
max_new_tokens = 20

for token in generate_text_basic_stream(model, input_tokens, max_new_tokens):
    token_id = token.squeeze(0).tolist()
    print(tokenizer.decode(token_id),
          end="",
          flush=True)



machine learning is a subset of artificial intelligence that is used to develop algorithms that can learn from data

In [5]:
### let's see how the eos_token_id works...

tokenizer.encode("<|endoftext|>")

[151643]

In [6]:
tokenizer.eos_token_id

151645

In [7]:
tokenizer.decode([151645]) ## this is the 

'<|im_end|>'

In [8]:
import torch
@torch.inference_mode()

def generate_text_streams(token_id, model,max_new_tokens, eos_token_id=None):
    model.eval()

    for _ in range(max_new_tokens):
        out = model(token_id) [:, -1]
        next_token = torch.argmax(out, dim=-1, keepdim=True)

        if (eos_token_id is not None and 
            torch.all(next_token == eos_token_id)):
            break

        yield next_token

        token_id = torch.cat([token_id, next_token], dim=1)

In [15]:
prompt = "what is the capital of france"
input_ids = torch.tensor(tokenizer.encode(prompt), device=device).unsqueeze(0)
max_new_tokens = 200

for token in generate_text_streams(input_ids, model, max_new_tokens, eos_token_id=tokenizer.eos_token_id):

    output_tokens = token.squeeze(0).tolist()
    print(tokenizer.decode(output_tokens),end="", flush=True)

?

The capital of France is Paris. However, it's important to note that the capital of France is also the capital of the French Republic, which is a constitutional monarchy in France. The capital of France is also the capital of the French Republic, which is a constitutional monarchy in France. The capital of France is also the capital of the French Republic, which is a constitutional monarchy in France. The capital of France is also the capital of the French Republic, which is a constitutional monarchy in France. The capital of France is also the capital of the French Republic, which is a constitutional monarchy in France. The capital of France is also the capital of the French Republic, which is a constitutional monarchy in France. The capital of France is also the capital of the French Republic, which is a constitutional monarchy in France. The capital of France is also the capital of the French Republic, which is a constitutional monarchy in France. The capital of France is also th

### getting the stats, to check how fast the generation operations is completed

In [16]:
import warnings

def synchronize_accelerator():
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elif getattr(torch, "xpu", None) is not None and torch.xpu.is_available():
        torch.xpu.synchronize()


def generate_stats(output_token_ids, start_time, end_time):
    total_time = end_time - start_time
    print(f"\n\nTime: {total_time: .2f} sec")
    print(f"{int(output_token_ids.numel() / total_time)} token/sec")


    for name, backend in (("CUDA", getattr(torch, "cuda")), ("XPU", getattr(torch, "xpu", None))):
        if backend is not None and backend.is_available():

            ## check wheter we are currently using this backend
            device_type = output_token_ids.device.type

            if device_type != name.lower():
                warnings.warn(
                    f"{name} is not available, but the tensors are on"
                    f"{device_type}. Memory stats may be 0"
                )

            ## synchronize
            if hasattr(backend, "synchronize"):
                backend.synchronize()

            max_memory_bytes = backend.max_memory_allocated()
            max_memory_gb = max_memory_bytes / (1024 ** 3)
            print(f"Max {name} memory allocated: {max_memory_gb:.2f} GB")
            backend.reset_peak_memory_stats()


In [17]:
import time 
synchronize_accelerator()
start_time = time.time()

generated_ids = []
decoded_tokens = []

for token in generate_text_streams(input_ids, model, max_new_tokens, tokenizer.eos_token_id):
    output_token = token.squeeze(0).tolist()
    decoded_tokens.append(tokenizer.decode(output_token))

    next_token_id = token.squeeze(0)
    generated_ids.append(next_token_id) ## collect generated tokens

synchronize_accelerator()
end = time.time()

print("".join(decoded_tokens), end="")
output_token_ids = torch.cat(generated_ids, dim=0)

generate_stats(output_token_ids, start_time, end)

?

The capital of France is Paris. However, it's important to note that the capital of France is also the capital of the French Republic, which is a constitutional monarchy in France. The capital of France is also the capital of the French Republic, which is a constitutional monarchy in France. The capital of France is also the capital of the French Republic, which is a constitutional monarchy in France. The capital of France is also the capital of the French Republic, which is a constitutional monarchy in France. The capital of France is also the capital of the French Republic, which is a constitutional monarchy in France. The capital of France is also the capital of the French Republic, which is a constitutional monarchy in France. The capital of France is also the capital of the French Republic, which is a constitutional monarchy in France. The capital of France is also the capital of the French Republic, which is a constitutional monarchy in France. The capital of France is also th

### ok using kv cache, to speed up the inference

In [ ]:
from base_model.qwen import KVCache

@torch.inference_mode()

def generate_text_streams_with_kv_cache(input_ids, model, max_generation_tokens, eos_token_id):
    model.eval()
    cache = KVCache(n_layers=model.cfg["n_layers"], max_seq_len=input_ids.shape[1] + max_generation_tokens)
    model.reset_kv_cache()  # Reset position tracker to 0

    # Prefill: process entire prompt at once
    out = model(input_ids, cache=cache)[:, -1]  # Get logits for last token

    for step in range(max_generation_tokens):
        next_token = torch.argmax(out, dim=-1, keepdim=True)

        if (eos_token_id is not None and
            torch.all(next_token == eos_token_id)):
            break

        yield next_token
        if step == max_generation_tokens - 1:
            break

        # Decoding: process only the new token with cached K,V
        out = model(next_token, cache=cache)[:, -1]


In [20]:
## now stats --- with the kv cache
synchronize_accelerator()
start_time = time.time()
generated_ids = []
decoded_tokens = []


for token in generate_text_streams_with_kv_cache(input_ids, model, max_new_tokens, tokenizer.eos_token_id):
    output_token = token.squeeze(0).tolist()

    decoded_tokens.append(tokenizer.decode(output_token))

    next_token = token.squeeze(0) ### to add to list --in the next step
    generated_ids.append(next_token)

synchronize_accelerator()
end_time = time.time()
print("".join(decoded_tokens), end="")
output_token_tensor = torch.cat(generated_ids, dim=0)
generate_stats(output_token_tensor, start_time, end_time)

?

The capital of France is Paris. However, it's important to note that the capital of France is also the capital of the French Republic, and the capital of the French government. The capital of France is Paris, and it is the seat of the French government. The capital of France is also the capital of the French Republic, and the capital of the French government. The capital of France is also the capital of the French Republic, and the capital of the French government. The capital of France is also the capital of the French Republic, and the capital of the French government. The capital of France is also the capital of the French Republic, and the capital of the French government. The capital of France is also the capital of the French Republic, and the capital of the French government. The capital of France is also the capital of the French Republic, and the capital of the French government. The capital of France is also the capital of the French Republic, and the capital of the French